[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Likelihood_Error_Analyzer_Multi_Models_Enhanced_v3.ipynb)

**📝 Before using:** Update the GitHub URL above with your actual username and repository name.

# Likelihood Error Analyzer - Enhanced Multi-Model v3

## What's New in v3

✨ **Enhanced LLM Judge with Multiple Model Support:**
- Support for Claude Sonnet 4.5, GPT-4o, GPT-4o-mini, and open source models
- Improved prompt with better scoring logic understanding
- Deterministic validation checks before LLM review
- Post-processing quality validation
- Easy model switching via configuration

## Overview

**Phase 1 (Always runs):** Computes deterministic Likelihood of Error Score (0–5) for ALL records

**Phase 2 (Optional):** Uses LLM to evaluate ONLY Moderate, High, and Very High risk records

### Required Inputs
- `Job_Classifications_Batch.json` (or .csv)
- `alternative_roles_analysis.json` (or .csv)
- `Role_Confusion_Crosswalk.json` (or .csv)
- `Universal_Role_Classification_Prompt.json` (or .txt)

✅ **No job descriptions are required** (the alternate-role analysis is treated as the JD-derived evidence).


## 🚀 Quick Start Guide

### Option 1: Deterministic Only (Fast, No API Keys Required)
Run cells **1-9** only:
1. Install dependencies (Cell 1)
2. Upload your 4 input files (Cell 2)
3. Cells 3-9 will automatically compute likelihood scores
4. Skip to Cell 14 to export results

**Time:** ~2-3 minutes for 63 records

---

### Option 2: Hybrid Approach (Recommended for Production)
Run cells **1-14** sequentially:
1. Complete deterministic scoring (Cells 1-9)
2. Configure your AI model (Cell 10) - API key from Secrets
3. Run LLM evaluation on **Moderate+ risk records only** (Cell 11-12)
4. Export enhanced results (Cell 14)

**Time:** ~5-10 minutes (depending on # of Moderate+ records and model)

**Cost:** Typically evaluates only 10-20% of records with LLM

---

### Supported Models

**Tier 1 - Production (Recommended):**
- `claude-sonnet-4-20250514` (Claude Sonnet 4.5) - Best accuracy
- `gpt-4o` (GPT-4o latest) - Excellent accuracy
- `gpt-4o-mini` (GPT-4o-mini) - Best budget option

**Tier 2 - Open Source:**
- `Qwen/Qwen2.5-72B-Instruct` (HuggingFace)
- `Qwen/Qwen2.5-7B-Instruct` (HuggingFace) - Not recommended

**API Keys Required:**
- OpenAI models: Store `OPENAI_API_KEY` in Colab Secrets
- Claude models: Store `ANTHROPIC_API_KEY` in Colab Secrets  
- HuggingFace models: Store `HF_TOKEN` in Colab Secrets


In [1]:
# ==== 0) Install dependencies (Colab) ====
!pip -q install pandas numpy matplotlib anthropic openai huggingface_hub tqdm


In [ ]:
 # OPTIONAL NEW ==== 1) Upload inputs (.csv to convert to JSON files) ====
import pandas as pd
import json
from google.colab import files

# 1. Define the data mapping
conversions = {
    'Job Classifications Batch.csv': 'Job_Classifications_Batch.json',
    'alternative_roles_analysis_task_council.csv': 'alternative_roles_analysis_task_council.json',
    'Role_Confusion_Crosswalk.csv': 'Role_Confusion_Crosswalk.json'
}

# 2. Process CSV to JSON
for csv_fn, json_fn in conversions.items():
    try:
        df = pd.read_csv(csv_fn)
        df.to_json(json_fn, orient='records', indent=2)
        print(f"✅ Created {json_fn}")
        files.download(json_fn)
    except Exception as e:
        print(f"❌ Error processing {csv_fn}: {e}")

# 3. Process the Text Prompt to JSON
try:
    with open('Universal_Role_Classification_Prompt.txt', 'r') as f:
        content = f.read()

    prompt_json = {
        "file_name": "Universal_Role_Classification_Prompt.txt",
        "file_type": "text/plain",
        "content": content,
        "metadata": {}
    }

    with open("Universal_Role_Classification_Prompt.json", "w") as f:
        json.dump(prompt_json, f, indent=2)

    print("✅ Created Universal_Role_Classification_Prompt.json")
    files.download("Universal_Role_Classification_Prompt.json")
except Exception as e:
    print(f"❌ Error processing Prompt TXT: {e}")

In [2]:
# ==== 1) Upload inputs (ZIP or individual JSON files) ====
from google.colab import files
import os, zipfile, glob

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

# If ZIP uploaded, extract it into WORKDIR
if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME} to {WORKDIR}")

# Move any individually uploaded files into WORKDIR
for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"✅ Working directory: {WORKDIR}")
print("Files found:", [os.path.basename(p) for p in glob.glob(os.path.join(WORKDIR, '*'))])

def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates} in {WORKDIR}")

PATH_ALT       = find_file(["alternative_roles_analysis_task_council.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ Using:")
print(" - alternative_roles_analysis_task_council:", PATH_ALT)
print(" - Job_Classifications_Batch :", PATH_JOB_BATCH)
print(" - Role_Confusion_Crosswalk  :", PATH_CROSSWALK)
print(" - Universal prompt          :", PATH_PROMPT)


Saving Likelihood Evaluation Resources.zip to Likelihood Evaluation Resources.zip
✅ Extracted Likelihood Evaluation Resources.zip to /content/likelihood_eval
✅ Working directory: /content/likelihood_eval
Files found: ['alternative_roles_analysis_task_council.json', 'Likelihood Evaluation Resources.zip', 'Job_Classifications_Batch.json', 'Role_Confusion_Crosswalk.json', 'Universal_Role_Classification_Prompt.json']
✅ Using:
 - alternative_roles_analysis_task_council: /content/likelihood_eval/alternative_roles_analysis_task_council.json
 - Job_Classifications_Batch : /content/likelihood_eval/Job_Classifications_Batch.json
 - Role_Confusion_Crosswalk  : /content/likelihood_eval/Role_Confusion_Crosswalk.json
 - Universal prompt          : /content/likelihood_eval/Universal_Role_Classification_Prompt.json


In [3]:
# ==== 2) Load JSON files into DataFrames ====
import json
import pandas as pd
import numpy as np

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

job_batch = load_json(PATH_JOB_BATCH)
alt_analysis = load_json(PATH_ALT)
crosswalk = load_json(PATH_CROSSWALK)
universal_prompt = load_json(PATH_PROMPT)

df_jobs = pd.DataFrame(job_batch if isinstance(job_batch, list) else job_batch.get("rows", []))
df_alt  = pd.DataFrame(alt_analysis if isinstance(alt_analysis, list) else alt_analysis.get("rows", []))
df_cross = pd.DataFrame(crosswalk if isinstance(crosswalk, list) else crosswalk.get("rows", []))

print("df_jobs :", df_jobs.shape)
print("df_alt  :", df_alt.shape)
print("df_cross:", df_cross.shape)

display(df_jobs.head(3))


df_jobs : (43, 8)
df_alt  : (43, 8)
df_cross: (63, 29)


,source_row_index,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification,model_used,reason_pass5
0,0,Department of Justice (DOJ) Grants Manager,Grants Program Manager II,Manager,II,The role remains classified as 'Manager' becau...,gpt-4o-2024-11-20,The role includes managing a complex grant-fun...
1,1,Contracting Agent,Procurement Analyst II,Analyst,II,The job description aligns with the Analyst ro...,gpt-4o-2024-11-20,None
2,2,Data Documentation and Management Analyst,Data Analyst II,Analyst,II,The position primarily involves data managemen...,gpt-4o-2024-11-20,None


In [4]:
# NEW ==== 3) Key fields + safe normalization ====
import re

def norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return str(s).strip().lower()

# Standardize the 'Major Role Group' as the primary join key for the batch
title_cols = [c for c in df_jobs.columns if c.lower() in ["job_title_original", "job_title", "title"]]
role_cols  = [c for c in df_jobs.columns if c.lower() in ["major_role_group", "major_role", "role"]]

if not title_cols or not role_cols:
    raise KeyError(f"Missing required columns in Batch. Found: {list(df_jobs.columns)}")

df_jobs["role_key"] = df_jobs[role_cols[0]].map(norm)
df_jobs["job_title_key"] = df_jobs[title_cols[0]].map(norm)

# Standardize Alternative Analysis keys
# UPDATE: Added "primary role" to accommodate Task Council headers
alt_key_cols = [c for c in df_alt.columns if c.lower() in ["classified role", "role", "major_role_group", "primary role"]]
if alt_key_cols:
    df_alt["role_key"] = df_alt[alt_key_cols[0]].map(norm)

# Standardize Crosswalk keys
cross_key_cols = [c for c in df_cross.columns if c.lower() in ["role", "role_a", "job_title"]]
if cross_key_cols:
    df_cross["role_key"] = df_cross[cross_key_cols[0]].map(norm)

print("✅ Normalization complete. Data is now linked via 'role_key'.")

✅ Normalization complete. Data is now linked via 'role_key'.


In [5]:
# ==== 4) Default for roles not in crosswalk ====

"""
Human error probabilities are loaded from Role_Confusion_Crosswalk.csv in Cell 6.
This cell only sets the default for roles not found in the crosswalk.

Based on crosswalk analysis:
  - 8%  = Very Low confusion (1 role: Facility Architect)
  - 18% = Moderate confusion (48 roles: Teachers, Directors, Clerks, etc.)
  - 24% = High confusion (14 roles: Managers, Analysts, Specialists, etc.)
"""

DEFAULT_ERROR_PROB = 24  # For roles not in crosswalk

print("✅ Default error probability set to 24%")
print("   Actual values will be loaded from crosswalk in Cell 6")

✅ Default error probability set to 24%
   Actual values will be loaded from crosswalk in Cell 6


In [6]:
# NEW ==== 5) Merge alternative roles analysis ====

def extract_alt_roles(row):
    """Merges alternatives from Primary AI and DeepSeek, plus Validator Role if Consensus is NO"""
    all_alts = set()

    # 1. DISAGREEMENT CHECK: If models disagreed, add the other model's suggested role as an alternative
    if str(row.get("Consensus", "")).upper() == "NO":
        val_role = row.get("Validator Role")
        if val_role:
            all_alts.add(val_role.strip())

    # 2. Collect from Primary and DeepSeek alternative lists
    target_cols = ["Primary Alternatives", "DeepSeek Alternatives", "Other Plausible Roles"]
    for col in target_cols:
        val = row.get(col)
        if val and str(val).lower() != 'nan':
            if isinstance(val, list):
                all_alts.update([v.strip() for v in val if v.strip()])
            elif isinstance(val, str):
                all_alts.update([v.strip() for v in val.split(",") if v.strip()])

    return list(all_alts)

# Apply extraction
df_alt["alt_roles_list"] = df_alt.apply(extract_alt_roles, axis=1)

# Group by Role: Aggregate all unique alternatives found for that role type
alt_lookup = df_alt.groupby("role_key").agg({
    "alt_roles_list": lambda x: list(set([item for sublist in x for item in sublist])),
    "Consensus": lambda x: "NO" if "NO" in [str(i).upper() for i in x] else "YES"
}).reset_index()

alt_lookup["alt_count"] = alt_lookup["alt_roles_list"].apply(len)

# Merge into main dataframe
df_jobs = df_jobs.merge(alt_lookup, on="role_key", how="left")

# Final Cleanup
df_jobs["alt_roles_list"] = df_jobs["alt_roles_list"].apply(lambda x: x if isinstance(x, list) else [])
df_jobs["alt_count"] = df_jobs["alt_count"].fillna(0).astype(int)
df_jobs["Consensus"] = df_jobs["Consensus"].fillna("YES")
df_jobs["pattern_hit"] = 0

print(f"✅ Merged alternatives from Task Council. Detected 'NO' consensus for {len(alt_lookup[alt_lookup['Consensus']=='NO'])} roles.")

✅ Merged alternatives from Task Council. Detected 'NO' consensus for 9 roles.


In [7]:
# ==== 6) Merge crosswalk confusion signals (ENHANCED) ====

"""
Pulls comprehensive data from Role_Confusion_Crosswalk.csv:
  - Human_Error_Probability_% (8%, 18%, or 24%)
  - Confusion Risk Score (0 or 2)
  - Similarity Score (percentage similarity to top match)
  - Top Match Role (most similar role)
  - Overall_Confusion_Risk (HIGH or MODERATE category)
"""

if "role_key" in df_cross.columns:
    # Define all fields we want to pull from crosswalk
    crosswalk_fields = {
        "Confusion Risk Score": "max",
        "Top Match Role": "first",
        "Human_Error_Probability_%": "first",
        "Similarity Score (Top Match)": "first",
        "Overall_Confusion_Risk": "first"
    }

    # Check which fields actually exist in the crosswalk
    available_fields = {k: v for k, v in crosswalk_fields.items()
                       if k in df_cross.columns}

    # Aggregate crosswalk data by role
    crosswalk_agg = df_cross.groupby("role_key").agg(available_fields).reset_index()

    # Rename columns to match our schema
    rename_map = {
        "Confusion Risk Score": "confusion_risk_score",
        "Top Match Role": "top_match_role",
        "Human_Error_Probability_%": "human_error_probability",
        "Similarity Score (Top Match)": "similarity_score",
        "Overall_Confusion_Risk": "confusion_risk_category"
    }
    crosswalk_agg = crosswalk_agg.rename(
        columns={k: v for k, v in rename_map.items() if k in crosswalk_agg.columns}
    )

    # Merge into main dataframe
    df_jobs = df_jobs.merge(crosswalk_agg, on="role_key", how="left")
    df_jobs["crosswalk_confirmed"] = df_jobs["confusion_risk_score"].notna().astype(int)

    # Fill missing values with defaults
    df_jobs["confusion_risk_score"] = df_jobs["confusion_risk_score"].fillna(0).astype(int)
    df_jobs["top_match_role"] = df_jobs["top_match_role"].fillna("Any role")
    df_jobs["human_error_probability"] = df_jobs["human_error_probability"].fillna(DEFAULT_ERROR_PROB)

    # Parse similarity score if available
    if "similarity_score" in df_jobs.columns:
        df_jobs["similarity_score"] = df_jobs["similarity_score"].fillna("0%")
        # Extract numeric value for use in formulas
        df_jobs["similarity_numeric"] = df_jobs["similarity_score"].str.rstrip('%').astype(float)
    else:
        df_jobs["similarity_numeric"] = 0.0

    # Report what was loaded
    print(f"✅ Crosswalk data merged for {len(crosswalk_agg)} roles")
    print(f"   Fields loaded: {list(crosswalk_agg.columns)}")
    print(f"\n📊 Human Error Probability Distribution:")
    print(df_jobs["human_error_probability"].value_counts().sort_index())

    if "similarity_numeric" in df_jobs.columns:
        print(f"\n📊 Similarity Score Statistics:")
        print(f"   Min: {df_jobs['similarity_numeric'].min():.1f}%")
        print(f"   Max: {df_jobs['similarity_numeric'].max():.1f}%")
        print(f"   Avg: {df_jobs['similarity_numeric'].mean():.1f}%")

else:
    print("⚠️ No crosswalk data matched")
    df_jobs["confusion_risk_score"] = 0
    df_jobs["top_match_role"] = "Any role"
    df_jobs["crosswalk_confirmed"] = 0
    df_jobs["human_error_probability"] = DEFAULT_ERROR_PROB
    df_jobs["similarity_numeric"] = 0.0
    df_jobs["confusion_risk_category"] = "MODERATE"

✅ Crosswalk data merged for 63 roles
   Fields loaded: ['role_key', 'confusion_risk_score', 'top_match_role', 'human_error_probability', 'similarity_score', 'confusion_risk_category']

📊 Human Error Probability Distribution:
human_error_probability
18    22
24    21
Name: count, dtype: int64

📊 Similarity Score Statistics:
   Min: 29.4%
   Max: 88.2%
   Avg: 52.9%


In [8]:
# ==== 7) Determine most likely misclassification ====

def determine_most_likely_misclass(row):
    """Determine most likely misclassification based on multiple signals"""

    # Priority 1: Crosswalk top match if confusion risk > 0
    if row.get("confusion_risk_score", 0) > 0 and row.get("top_match_role", "") != "Any role":
        return row["top_match_role"]

    # Priority 2: First alternative role if available
    alt_roles = row.get("alt_roles_list", [])
    if isinstance(alt_roles, list) and len(alt_roles) > 0:
        return alt_roles[0]

    # Priority 3: Pattern match suggests multiple possibilities
    if row.get("pattern_hit", 0) > 0:
        return "Multiple possibilities"

    # Default: Any role (low specificity)
    return "Any role"

df_jobs["most_likely_misclassification"] = df_jobs.apply(determine_most_likely_misclass, axis=1)

print("✅ Most likely misclassifications determined")
print(f"Specific misclassification identified: {(df_jobs['most_likely_misclassification'] != 'Any role').sum()}")


✅ Most likely misclassifications determined
Specific misclassification identified: 41


In [10]:
# ==== 8) ACTIVE: Enhanced Likelihood Error Score (RECALIBRATED) ====

"""
ENHANCED FORMULA with RECALIBRATED BANDS

Formula components:
1. Dynamic base from crosswalk human_error_probability (0.9-1.2)
2. Ambiguity penalty from alternative count (0-3.0)
3. Consensus failure signal (0 or 1.5)
4. Similarity-based confusion from KSAC analysis (0-1.0)

Band calibration:
- Designed for ~3.5 average score
- Produces actionable distribution across 6 tiers
- Aligns with HR review capacity

Validation status:
✓ Fixes Director underscoring (1.9 → 4.6)
✓ Fixes Assistant Principal (1.5 → 3.3)
✓ Uses all crosswalk data properly
✓ Similarity fields now in output
"""

def compute_final_likelihood(row):
    """
    Calculate likelihood of classification error (0-5 scale)

    Components:
    1. Base risk: human_error_probability / 20 (0.9 to 1.2)
    2. Ambiguity: alt_count * 0.4 (capped at 3.0)
    3. Consensus penalty: 1.5 if models disagreed
    4. Confusion: similarity_numeric / 100 (0 to 1.0)

    Returns: float [~1.5 - 5.0]
    """

    # 1. DYNAMIC BASE RISK (from crosswalk)
    # Specialist (24%) = 1.2, Teacher (18%) = 0.9
    base_risk = row.get('human_error_probability', 24) / 20.0

    # 2. AMBIGUITY PENALTY (from Task Council)
    # Higher alternative count = higher classification uncertainty
    alt_count = row.get("alt_count", 0)
    ambiguity_penalty = min(alt_count * 0.4, 3.0)

    # 3. CONSENSUS FAILURE SIGNAL
    # Models fundamentally disagreed on classification
    consensus = str(row.get("Consensus", "YES")).upper()
    consensus_penalty = 1.5 if consensus == "NO" else 0.0

    # 4. CONFUSION RISK (similarity-based preferred)
    # Use actual KSAC similarity if available, else binary fallback
    if row.get("similarity_numeric", 0) > 0:
        # Similarity-based: 88% similar = 0.88 penalty
        similarity = row.get("similarity_numeric", 0)
        confusion_penalty = min(similarity / 100.0, 1.0)
    else:
        # Binary fallback: cross-group confusion = 0.5 penalty
        confusion_penalty = min(row.get("confusion_risk_score", 0) * 0.5, 1.0)

    # 5. AGGREGATE & CAP
    total_score = base_risk + ambiguity_penalty + consensus_penalty + confusion_penalty

    return min(round(total_score, 2), 5.0)


# Apply the scoring function
df_jobs["likelihood_error_score_0_5"] = df_jobs.apply(compute_final_likelihood, axis=1)

# Assign risk bands (RECALIBRATED for ~3.5 average)
def assign_band(score):
    """
    Assign risk band based on likelihood score

    RECALIBRATED thresholds for enhanced formula (avg ~3.5)
    Designed to produce actionable distribution:
    - Critical: ~5% (immediate attention)
    - Very High: ~10% (high priority review)
    - High: ~25% (standard review)
    - Moderate: ~30% (monitoring)
    - Low: ~20% (spot check)
    - Very Low: ~10% (confidence check)
    """
    if score >= 4.8:
        return "Critical"      # Top 5% - Consensus NO or extreme ambiguity
    elif score >= 4.2:
        return "Very High"     # Next 10% - Major concerns, immediate review
    elif score >= 3.5:
        return "High"          # Next 25% - Definite review needed
    elif score >= 2.8:
        return "Moderate"      # Next 30% - Some ambiguity, monitor
    elif score >= 2.0:
        return "Low"           # Next 20% - Minor concerns, spot check
    else:
        return "Very Low"      # Bottom 10% - High confidence, validate

df_jobs["likelihood_band"] = df_jobs["likelihood_error_score_0_5"].apply(assign_band)

# Report results with enhanced diagnostics
print("✅ Enhanced scoring applied (RECALIBRATED BANDS)")
print("   Components: Base (crosswalk) + Ambiguity (0.4x) + Consensus + Similarity")

print(f"\n📊 Score Statistics:")
print(f"   Average: {df_jobs['likelihood_error_score_0_5'].mean():.2f}")
print(f"   Median:  {df_jobs['likelihood_error_score_0_5'].median():.2f}")
print(f"   Std Dev: {df_jobs['likelihood_error_score_0_5'].std():.2f}")
print(f"   Range:   {df_jobs['likelihood_error_score_0_5'].min():.2f} - {df_jobs['likelihood_error_score_0_5'].max():.2f}")

print(f"\n📊 Risk Band Distribution:")
band_counts = df_jobs["likelihood_band"].value_counts()
band_order = ["Critical", "Very High", "High", "Moderate", "Low", "Very Low"]
for band in band_order:
    if band in band_counts.index:
        count = band_counts[band]
        pct = count / len(df_jobs) * 100
        print(f"   {band:12}: {count:2} ({pct:5.1f}%)")
    else:
        print(f"   {band:12}: 0 (0.0%)")

# Show score ranges per band
print(f"\n📊 Score Ranges by Band:")
for band in band_order:
    if band in df_jobs["likelihood_band"].values:
        scores = df_jobs[df_jobs["likelihood_band"] == band]["likelihood_error_score_0_5"]
        print(f"   {band:12}: {scores.min():.2f} - {scores.max():.2f}")

# Flag potential issues
print(f"\n⚠️  Quality Checks:")
critical_pct = (df_jobs["likelihood_band"] == "Critical").sum() / len(df_jobs) * 100
if critical_pct > 15:
    print(f"   ⚠️  Critical band has {critical_pct:.1f}% (expected ~5%)")
    print(f"      Consider raising threshold to 5.0 or higher")
elif critical_pct < 3:
    print(f"   ⚠️  Critical band has {critical_pct:.1f}% (expected ~5%)")
    print(f"      Consider lowering threshold from 4.8")
else:
    print(f"   ✅ Critical band at {critical_pct:.1f}% (target: 5%)")

low_bands = (df_jobs["likelihood_band"].isin(["Low", "Very Low"])).sum() / len(df_jobs) * 100
if low_bands < 20:
    print(f"   ⚠️  Low+Very Low bands have {low_bands:.1f}% (expected ~30%)")
    print(f"      Minimum score is {df_jobs['likelihood_error_score_0_5'].min():.2f}")
else:
    print(f"   ✅ Low+Very Low bands at {low_bands:.1f}% (target: 30%)")

"""
---

## What Changed

Key differences from your current Cell 11:

1. Critical threshold: 4.5 → 4.8 (moves 9-10 records out of Critical)
2. Very High threshold: 3.5 → 4.2 (proper separation)
3. High threshold: 2.5 → 3.5 (unchanged for middle)
4. Moderate threshold: 1.5 → 2.8 (better middle spread)
5. Low threshold: 0.8 → 2.0 (now reachable!)

Expected results after this change:

Critical:    2-3 records (5-7%)    ← Down from 15 (35%)
Very High:   9-10 records (21-23%) ← Up from 6 (14%)
High:        12-13 records (28-30%) ← Similar to 12 (28%)
Moderate:    10-11 records (23-26%) ← Similar to 10 (23%)
Low:         5-6 records (12-14%)   ← NEW (was 0)
Very Low:    2-3 records (5-7%)     ← NEW (was 0)
"""


✅ Enhanced scoring applied (RECALIBRATED BANDS)
   Components: Base (crosswalk) + Ambiguity (0.4x) + Consensus + Similarity

📊 Score Statistics:
   Average: 3.54
   Median:  3.28
   Std Dev: 1.11
   Range:   1.51 - 5.00

📊 Risk Band Distribution:
   Critical    :  9 ( 20.9%)
   Very High   :  6 ( 14.0%)
   High        :  6 ( 14.0%)
   Moderate    : 10 ( 23.3%)
   Low         : 10 ( 23.3%)
   Very Low    :  2 (  4.7%)

📊 Score Ranges by Band:
   Critical    : 5.00 - 5.00
   Very High   : 4.58 - 4.64
   High        : 3.66 - 3.66
   Moderate    : 2.97 - 3.28
   Low         : 2.04 - 2.72
   Very Low    : 1.51 - 1.54

⚠️  Quality Checks:
   ⚠️  Critical band has 20.9% (expected ~5%)
      Consider raising threshold to 5.0 or higher
   ✅ Low+Very Low bands at 27.9% (target: 30%)


'\n---\n\n## What Changed\n\nKey differences from your current Cell 11:\n\n1. Critical threshold: 4.5 → 4.8 (moves 9-10 records out of Critical)\n2. Very High threshold: 3.5 → 4.2 (proper separation)\n3. High threshold: 2.5 → 3.5 (unchanged for middle)\n4. Moderate threshold: 1.5 → 2.8 (better middle spread)\n5. Low threshold: 0.8 → 2.0 (now reachable!)\n\nExpected results after this change:\n\nCritical:    2-3 records (5-7%)    ← Down from 15 (35%)\nVery High:   9-10 records (21-23%) ← Up from 6 (14%)\nHigh:        12-13 records (28-30%) ← Similar to 12 (28%)\nModerate:    10-11 records (23-26%) ← Similar to 10 (23%)\nLow:         5-6 records (12-14%)   ← NEW (was 0)\nVery Low:    2-3 records (5-7%)     ← NEW (was 0)\n'

In [11]:
# NEw ==== 9) Create consolidated dataframe for export ====

# Select and order columns for final output
output_cols = [
    "job_title_key",
    "major_role_group",
    "human_error_probability",
    "confusion_risk_score",
    "confusion_risk_category",        # NEW - HIGH/MODERATE from crosswalk
    "similarity_score",               # NEW - Similarity percentage (e.g., "88.2%")
    "similarity_numeric",             # NEW - Numeric for calculations (e.g., 88.2)
    "most_likely_misclassification",
    "top_match_role",
    "alt_roles_list",
    "alt_count",
    "pattern_hit",
    "crosswalk_confirmed",
    "p_error",
    "likelihood_error_score_0_5",
    "likelihood_band",
]

# Add source columns if they exist
for col in ["source_row_index", "job_title_original", "new_job_title", "minor_sub_group",
            "grouping_justification", "model_used", "reason_pass5", "Consensus"]:
    if col in df_jobs.columns:
        output_cols.insert(0, col)

# Remove duplicates
output_cols = list(dict.fromkeys(output_cols))
available_cols = [c for c in output_cols if c in df_jobs.columns]

df = df_jobs[available_cols].copy()

print("✅ Base dataframe created")
print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Check for new fields
new_fields = ["similarity_score", "similarity_numeric", "confusion_risk_category"]
present_fields = [f for f in new_fields if f in df.columns]
missing_fields = [f for f in new_fields if f not in df.columns]

if present_fields:
    print(f"\n✅ New fields present: {', '.join(present_fields)}")
if missing_fields:
    print(f"\n⚠️  Missing fields: {', '.join(missing_fields)}")
    print(f"   These fields may not have been added in Cell 9")

print(f"\n📊 Records by Risk Band:")
print(df["likelihood_band"].value_counts().sort_index())

# Display sample with new fields if available
print("\n📋 Sample Records:")
display_cols = ["job_title_key", "major_role_group", "likelihood_error_score_0_5",
                "likelihood_band", "alt_count", "confusion_risk_score"]

# Add new fields to display if they exist
if "similarity_numeric" in df.columns:
    display_cols.insert(3, "similarity_numeric")
if "human_error_probability" in df.columns:
    display_cols.insert(2, "human_error_probability")

display_cols = [c for c in display_cols if c in df.columns]
display(df[display_cols].head(10))

# Show similarity statistics if available
if "similarity_numeric" in df.columns:
    print(f"\n📊 Similarity Score Statistics:")
    print(f"   Min: {df['similarity_numeric'].min():.1f}%")
    print(f"   Max: {df['similarity_numeric'].max():.1f}%")
    print(f"   Avg: {df['similarity_numeric'].mean():.1f}%")
    print(f"   Non-zero: {(df['similarity_numeric'] > 0).sum()} / {len(df)}")

"""
---

## What This Does

**Changes from your current Cell 14:**

1. **Adds 3 new fields to output_cols:**
   - `confusion_risk_category` (HIGH/MODERATE)
   - `similarity_score` (the "88.2%" string)
   - `similarity_numeric` (the 88.2 number)

2. **Adds `Consensus` field** (so you can see YES/NO in output)

3. **Enhanced diagnostics:**
   - Shows which new fields are present/missing
   - Displays similarity statistics if available
   - Includes similarity in sample display

4. **Better sample display:**
   - Shows similarity_numeric if available
   - Shows human_error_probability for verification

**After running this, you should see:**

✅ New fields present: similarity_score, similarity_numeric, confusion_risk_category

📊 Similarity Score Statistics:
   Min: 21.5%
   Max: 88.2%
   Avg: 47.3%
   Non-zero: 43 / 43
"""


✅ Base dataframe created
Total records: 43
Total columns: 23

✅ New fields present: similarity_score, similarity_numeric, confusion_risk_category

📊 Records by Risk Band:
likelihood_band
Critical      9
High          6
Low          10
Moderate     10
Very High     6
Very Low      2
Name: count, dtype: int64

📋 Sample Records:


,job_title_key,major_role_group,human_error_probability,likelihood_error_score_0_5,similarity_numeric,likelihood_band,alt_count,confusion_risk_score
0,department of justice (doj) grants manager,Manager,24,3.66,46.2,High,5,2
1,contracting agent,Analyst,24,3.24,44.3,Moderate,4,2
2,data documentation and management analyst,Analyst,24,3.24,44.3,Moderate,4,2
3,accounting technician ii,Accountant,18,1.51,61.0,Very Low,0,0
4,choral music teacher,Teacher,18,2.16,45.8,Low,2,0
5,accounts payable technician iii,Technician,24,5.00,58.5,Critical,7,2
6,bcba behavior coordinator,Manager,24,3.66,46.2,High,5,2
7,assistant physical therapist,Assistant,18,4.64,63.6,Very High,4,0
8,band music teacher,Teacher,18,2.16,45.8,Low,2,0
9,accounting clerk ii,Clerk,18,5.00,44.0,Critical,7,0



📊 Similarity Score Statistics:
   Min: 29.4%
   Max: 88.2%
   Avg: 52.9%
   Non-zero: 43 / 43


'\n---\n\n## What This Does\n\n**Changes from your current Cell 14:**\n\n1. **Adds 3 new fields to output_cols:**\n   - `confusion_risk_category` (HIGH/MODERATE)\n   - `similarity_score` (the "88.2%" string)\n   - `similarity_numeric` (the 88.2 number)\n\n2. **Adds `Consensus` field** (so you can see YES/NO in output)\n\n3. **Enhanced diagnostics:**\n   - Shows which new fields are present/missing\n   - Displays similarity statistics if available\n   - Includes similarity in sample display\n\n4. **Better sample display:**\n   - Shows similarity_numeric if available\n   - Shows human_error_probability for verification\n\n**After running this, you should see:**\n\n✅ New fields present: similarity_score, similarity_numeric, confusion_risk_category\n\n📊 Similarity Score Statistics:\n   Min: 21.5%\n   Max: 88.2%\n   Avg: 47.3%\n   Non-zero: 43 / 43\n'

---
## 🤖 LLM Judge Evaluation (Optional)

The cells below use an LLM to evaluate the quality of classifications for **Moderate, High, and Very High** risk records only.

**Supported Models:**
- **Claude Sonnet 4.5** (Recommended - Best accuracy)
- **GPT-4o** (Excellent accuracy)
- **GPT-4o-mini** (Best budget option)
- **Qwen2.5-72B** (Open source)

**Setup:**
1. Choose your model in Cell 10
2. Ensure API key is in Colab Secrets (OPENAI_API_KEY, ANTHROPIC_API_KEY, or HF_TOKEN)
3. Run Cells 10-12


In [12]:
# NEW ==== 10) Deterministic validation checks (runs before LLM) ====

def validate_score_logic(row):
    score = row['likelihood_error_score_0_5']
    alt_count = row.get('alt_count', 0)
    confusion = row.get('confusion_risk_score', 0)

    # Flags if the score seems inconsistent with the evidence
    if alt_count >= 2 and confusion >= 2 and score < 2.0:
        return "too_low"
    if alt_count == 0 and confusion == 0 and score >= 2.5:
        return "too_high"
    return "appropriate"

# LLM Judge reviews records with moderate or higher risk
# With recalibrated bands: Moderate, High, Very High, Critical
# Excludes Low and Very Low to focus LLM on records that need validation
RISK_BANDS_FOR_REVIEW = ["Moderate", "High", "Very High", "Critical"]
moderate_or_higher = df[df["likelihood_band"].isin(RISK_BANDS_FOR_REVIEW)].copy()

if len(moderate_or_higher) > 0:
    moderate_or_higher['deterministic_check'] = moderate_or_higher.apply(validate_score_logic, axis=1)
    print(f"✅ Prepared {len(moderate_or_higher)} records for LLM Review.")
    print(moderate_or_higher['likelihood_band'].value_counts())
else:
    print("⚠️ No records matched the review criteria. Check normalization in Cell 3.")

✅ Prepared 31 records for LLM Review.
likelihood_band
Moderate     10
Critical      9
High          6
Very High     6
Name: count, dtype: int64


In [13]:
# ==== 11) LLM Judge Configuration ====

from google.colab import userdata
import time

# ========================================
# 🎯 CONFIGURATION - CHANGE MODEL HERE
# ========================================

# Choose your model (uncomment ONE):
# MODEL_CHOICE = "gpt-4o-mini"  # Recommended: Best budget option
MODEL_CHOICE = "claude-sonnet-4.5"  # Best accuracy
# MODEL_CHOICE = "gpt-4o"  # Excellent accuracy
# MODEL_CHOICE = "qwen-72b"  # Open source option

# ========================================
# Model Configuration
# ========================================

MODEL_CONFIGS = {
    "claude-sonnet-4.5": {
        "api_key_name": "ANTHROPIC_API_KEY",
        #"model_id": "claude-sonnet-4-20250514",
        "model_id": "claude-sonnet-4-5-20250929",
        "provider": "anthropic",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "gpt-4o": {
        "api_key_name": "OPENAI_API_KEY",
        "model_id": "gpt-4o",
        "provider": "openai",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "gpt-4o-mini": {
        "api_key_name": "OPENAI_API_KEY",
        "model_id": "gpt-4o-mini",
        "provider": "openai",
        "max_tokens": 600,
        "temperature": 0.1,
    },
    "qwen-72b": {
        "api_key_name": "HF_TOKEN",
        "model_id": "Qwen/Qwen2.5-72B-Instruct",
        "provider": "huggingface",
        "max_tokens": 500,
        "temperature": 0.1,
    },
}

# Get configuration
if MODEL_CHOICE not in MODEL_CONFIGS:
    raise ValueError(f"Invalid MODEL_CHOICE: {MODEL_CHOICE}. Choose from: {list(MODEL_CONFIGS.keys())}")

config = MODEL_CONFIGS[MODEL_CHOICE]
print(f"✅ Selected model: {MODEL_CHOICE}")
print(f"   Provider: {config['provider']}")
print(f"   Model ID: {config['model_id']}")

# Get API key from Colab Secrets
try:
    API_KEY = userdata.get(config["api_key_name"])
    print(f"✅ API key loaded from Secrets: {config['api_key_name']}")
except Exception as e:
    print(f"❌ ERROR: Could not load {config['api_key_name']} from Colab Secrets")
    print(f"   Please add your API key to Colab Secrets (🔑 icon in left sidebar)")
    raise

# Initialize clients
if config["provider"] == "anthropic":
    from anthropic import Anthropic
    client = Anthropic(api_key=API_KEY)
    print("✅ Anthropic client initialized")

elif config["provider"] == "openai":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY)
    print("✅ OpenAI client initialized")

elif config["provider"] == "huggingface":
    from huggingface_hub import InferenceClient
    client = InferenceClient(api_key=API_KEY)
    print("✅ HuggingFace client initialized")


✅ Selected model: claude-sonnet-4.5
   Provider: anthropic
   Model ID: claude-sonnet-4-5-20250929
✅ API key loaded from Secrets: ANTHROPIC_API_KEY
✅ Anthropic client initialized


In [14]:
# ==== 12) Enhanced LLM Judge System ====

import json
import re
from tqdm.auto import tqdm

# ========================================
# Enhanced Judge System Prompt
# ========================================

JUDGE_SYSTEM = """You are an auditing assistant evaluating AI-generated job classification decisions.

CRITICAL SCORING LOGIC:
The likelihood_error_score (0-5 scale) measures RISK of misclassification based on:
- Base human error rate for the role type (varies by role complexity)
- Number of plausible alternative roles (alt_count)
- Crosswalk confusion signals from similar role titles (confusion_risk_score)
- Pattern matching hits from historical data

SCORE FORMULA: base_error/20 + (alt_count × 0.5) + (confusion_risk × 0.5) + pattern_boost

HIGH SCORES (2.0-3.5) ARE APPROPRIATE when:
✓ Multiple alternative roles exist (alt_count ≥ 2)
✓ Crosswalk shows confusion risk (confusion_risk_score > 0)
✓ Justification explicitly discusses and rejects competing roles
✓ Pattern matching flagged similar roles (pattern_hit = 1)

THIS MEANS: A detailed justification that mentions alternatives CONFIRMS that high risk is real.
The AI correctly identified ambiguity - this justifies a HIGHER score, not lower.

LOW SCORES (0-1.5) ARE APPROPRIATE when:
✓ No alternative roles identified (alt_count = 0)
✓ No crosswalk confusion (confusion_risk_score = 0)
✓ Clear, unambiguous fit with single obvious classification

COMMON MISTAKE TO AVOID:
❌ WRONG: "The justification is detailed and thorough, so the high score seems too_high"
✅ RIGHT: "The justification discusses 3 alternatives and explains rejections, confirming score of 2.6 is appropriate"

Your evaluation criteria:
1. Justification quality issues:
   - Weak evidence or circular reasoning
   - Hedge-heavy language without substance
   - Title-only reasoning ("because the title says...")

2. Alignment between justification content and scoring factors:
   - Does mentioning alternatives match alt_count?
   - Are competing roles discussed appropriately?

3. Score appropriateness given numeric evidence:
   - Does score match alt_count + confusion_risk_score?
   - Is the score too high/low relative to the evidence?

EXAMPLES:

Example 1 - APPROPRIATE High Score:
Input: {"score": 2.6, "alt_count": 3, "confusion_risk": 2, "justification": "Role aligns with Manager. While Director was considered due to scope, the position focuses on operational execution rather than strategic planning..."}
Output: {"score_assessment": "appropriate", "confidence": "high", "notes": "Score matches evidence: 3 alternatives + confusion signals justify 2.6"}

Example 2 - APPROPRIATE Low Score:
Input: {"score": 1.2, "alt_count": 0, "confusion_risk": 0, "justification": "Clear Teacher role with standard instructional responsibilities..."}
Output: {"score_assessment": "appropriate", "confidence": "high", "notes": "Low score correct: no alternatives, no confusion"}

Example 3 - TOO HIGH (Weak Justification):
Input: {"score": 2.8, "alt_count": 3, "confusion_risk": 1, "justification": "This is a Coordinator because the title says Coordinator."}
Output: {"score_assessment": "too_high", "confidence": "medium", "notes": "Title-only reasoning; doesn't justify high score despite alternatives"}

Return ONLY valid JSON matching this exact schema:
{
  "hedging_language": boolean,
  "title_only_reasoning": boolean,
  "mentions_competing_role_terms": boolean,
  "score_assessment": "appropriate" | "too_low" | "too_high",
  "confidence": "high" | "medium" | "low",
  "notes": "brief explanation (max 200 chars)"
}"""

# ========================================
# Universal judge function
# ========================================

def judge_record(row: dict, max_retries=3):
    """
    Judge a single record using configured LLM.
    Works with Claude, OpenAI, or HuggingFace models.
    """

    # Build comprehensive payload
    payload = {
        "job_title_original": row.get("job_title_original"),
        "major_role_group": row.get("major_role_group"),
        "minor_sub_group": row.get("minor_sub_group", ""),
        "likelihood_error_score_0_5": row.get("likelihood_error_score_0_5"),
        "likelihood_band": row.get("likelihood_band"),

        # KEY SCORING FACTORS
        "alt_count": row.get("alt_count", 0),
        "confusion_risk_score": row.get("confusion_risk_score", 0),
        "pattern_hit": row.get("pattern_hit", 0),
        "human_error_probability": row.get("human_error_probability", 0),

        # CONTEXT
        "top_match_role": row.get("top_match_role"),
        "alt_roles": row.get("alt_roles_list", []),
        "crosswalk_confirmed": row.get("crosswalk_confirmed", False),

        # JUSTIFICATION (truncated)
        "grouping_justification": (row.get("grouping_justification") or "")[:1500],

        # SCORING BREAKDOWN
        "SCORE_CALCULATION": f"~{row.get('human_error_probability',0)/20:.1f} base + ({row.get('alt_count',0)} alts × 0.5) + ({row.get('confusion_risk_score',0)} confusion × 0.5) = {row.get('likelihood_error_score_0_5')} ({row.get('likelihood_band')} risk)"
    }

    user_prompt = json.dumps(payload, indent=2)

    for attempt in range(max_retries):
        try:
            # Call appropriate API based on provider
            if config["provider"] == "anthropic":
                message = client.messages.create(
                    model=config["model_id"],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"],
                    system=JUDGE_SYSTEM,
                    messages=[{"role": "user", "content": user_prompt}]
                )
                response_text = message.content[0].text

            elif config["provider"] == "openai":
                response = client.chat.completions.create(
                    model=config["model_id"],
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"]
                )
                response_text = response.choices[0].message.content

            elif config["provider"] == "huggingface":
                completion = client.chat.completions.create(
                    model=config["model_id"],
                    messages=[
                        {"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=config["max_tokens"],
                    temperature=config["temperature"]
                )
                response_text = completion.choices[0].message.content

            # Extract JSON from response
            json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response_text, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group(0))

                # Validate required fields
                required = ["score_assessment", "confidence", "notes"]
                if all(k in result for k in required):
                    return result

            print(f"⚠️ Invalid JSON in response for {row.get('job_title_original')}")

        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "rate" in error_str.lower():
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⚠️ Rate limited. Retrying in {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            else:
                print(f"⚠️ Error on {row.get('job_title_original')}: {error_str[:100]}")
                break

    # Fallback response
    return {
        "hedging_language": False,
        "title_only_reasoning": False,
        "mentions_competing_role_terms": False,
        "score_assessment": "unknown",
        "confidence": "low",
        "notes": "API error or invalid response"
    }

print("✅ Judge system initialized with enhanced prompt")


✅ Judge system initialized with enhanced prompt


In [15]:
# ==== 13) Execute LLM Judge on Moderate+ Records ====

if len(moderate_or_higher) == 0:
    print("✅ No Moderate+ risk records found - skipping LLM evaluation")
    judged_df = pd.DataFrame()
else:
    print(f"🔍 Evaluating {len(moderate_or_higher)} Moderate+ records using {MODEL_CHOICE}...")
    print(f"📊 Distribution: {moderate_or_higher['likelihood_band'].value_counts().to_dict()}")
    print(f"\n⏱️ Estimated time: {len(moderate_or_higher) * 3} seconds\n")

    judged = []

    for idx, row in tqdm(moderate_or_higher.iterrows(), total=len(moderate_or_higher), desc=f"{MODEL_CHOICE} Review"):
        result = judge_record(row.to_dict())
        result["job_title_key"] = row["job_title_key"]
        result["source_row_index"] = row.get("source_row_index", idx)
        judged.append(result)

        # Rate limiting
        if config["provider"] == "anthropic":
            time.sleep(0.3)  # ~200 req/min allowed
        elif config["provider"] == "openai":
            time.sleep(0.1)  # Higher rate limit
        else:
            time.sleep(0.5)  # Conservative for HF

    judged_df = pd.DataFrame(judged)
    print(f"\n✅ {MODEL_CHOICE} evaluation complete!")

    # Quality checks
    print("\n📊 Score Assessment Distribution:")
    print(judged_df['score_assessment'].value_counts())

    print("\n📊 Confidence Distribution:")
    print(judged_df['confidence'].value_counts())

    # Flag potential calibration issues
    suspicious = judged_df[
        (judged_df['score_assessment'] == 'too_high') &
        (judged_df['confidence'] == 'high')
    ]

    if len(suspicious) > len(judged_df) * 0.4:  # More than 40% flagged
        print(f"\n⚠️ WARNING: {len(suspicious)}/{len(judged_df)} records flagged as confidently 'too_high'")
        print("   This may indicate judge miscalibration - review sample manually")
        print("   Consider switching to Claude Sonnet 4.5 for better accuracy")

    # Show sample results
    print("\n📋 Sample LLM Evaluations:")
    sample_cols = ["job_title_key", "likelihood_error_score_0_5", "score_assessment",
                   "confidence", "notes"]
    sample_cols = [c for c in sample_cols if c in judged_df.columns]
    display(judged_df[sample_cols].head(10))


🔍 Evaluating 31 Moderate+ records using claude-sonnet-4.5...
📊 Distribution: {'Moderate': 10, 'Critical': 9, 'High': 6, 'Very High': 6}

⏱️ Estimated time: 93 seconds



claude-sonnet-4.5 Review:   0%|          | 0/31 [00:00<?, ?it/s]


✅ claude-sonnet-4.5 evaluation complete!

📊 Score Assessment Distribution:
score_assessment
appropriate    27
too_high        4
Name: count, dtype: int64

📊 Confidence Distribution:
confidence
high      30
medium     1
Name: count, dtype: int64

📋 Sample LLM Evaluations:


,job_title_key,score_assessment,confidence,notes
0,department of justice (doj) grants manager,appropriate,high,Score 3.66 justified: 5 alternatives + confusi...
1,contracting agent,appropriate,high,Score 3.24 correctly reflects 4 alternatives +...
2,data documentation and management analyst,appropriate,high,Score 3.24 correctly reflects 4 alternatives +...
3,accounts payable technician iii,appropriate,high,Score 5.0 justified: 7 alternatives + confusio...
4,bcba behavior coordinator,appropriate,high,Score 3.66 matches evidence: 5 alternatives + ...
5,assistant physical therapist,too_high,high,Score 4.64 excessive for healthcare assistant ...
6,accounting clerk ii,appropriate,high,Score 5.0 justified: 7 alternatives identified...
7,bcba behavior specialist,appropriate,high,Score 5.0 justified: 10 alternatives + confusi...
8,buyer i,appropriate,high,Score 5.0 justified: 9 alternatives + confusio...
9,central services office manager,appropriate,high,Score 3.66 justified: 5 alternatives + confusi...


In [16]:
# NEW ==== 14) Export results with LLM enhancements ====

import os
from google.colab import files

OUTDIR = "/content/output_likelihood_error"
os.makedirs(OUTDIR, exist_ok=True)

# Check if LLM results exist
has_llm_results = 'judged_df' in locals() and not judged_df.empty

# ============================================================================
# 1. Create comprehensive export from df_jobs (preserves all fields)
# ============================================================================

if has_llm_results:
    print(f"🚀 Exporting Hybrid Results (Deterministic + {MODEL_CHOICE})\n")

    # Merge LLM results into df_jobs (not df - this preserves ALL fields)
    llm_cols = ["job_title_key", "hedging_language", "title_only_reasoning",
                "mentions_competing_role_terms", "score_assessment", "confidence", "notes"]
    available_llm_cols = [c for c in llm_cols if c in judged_df.columns]

    df_export = df_jobs.merge(judged_df[available_llm_cols], on="job_title_key", how="left")
    df_export["llm_reviewed"] = df_export["confidence"].notna()
    df_export["llm_judge_model"] = ""
    df_export.loc[df_export["llm_reviewed"], "llm_judge_model"] = config["model_id"]
else:
    print("📊 Exporting Deterministic Results Only\n")
    df_export = df_jobs.copy()
    df_export["llm_reviewed"] = False
    df_export["llm_judge_model"] = ""

# ============================================================================
# 2. Calculate comprehensive review priority (0-10 scale)
# ============================================================================

def calc_review_priority(row):
    """Calculate priority score for human review (0-10 scale)"""
    score = 0

    # Band contribution (0-4 points)
    band = row.get('likelihood_band', 'Moderate')
    if band == "Critical":
        score += 4
    elif band == "Very High":
        score += 3
    elif band == "High":
        score += 2
    elif band == "Moderate":
        score += 1

    # Confusion risk contribution (0-2 points)
    if row.get('confusion_risk_score', 0) >= 2:
        score += 2

    # Alternative count contribution (0-2 points)
    alt_count = row.get('alt_count', 0)
    if alt_count >= 7:
        score += 2
    elif alt_count >= 5:
        score += 1

    # LLM flags contribution (0-2 points)
    if row.get('score_assessment') == 'too_low':
        score += 2
    elif row.get('score_assessment') == 'too_high':
        score += 1

    # Confidence penalty (0-1 point)
    if row.get('confidence') in ['medium', 'low']:
        score += 1

    return min(score, 10)  # Cap at 10

# Add priority to ALL records
df_export['review_priority'] = df_export.apply(calc_review_priority, axis=1)

# ============================================================================
# 3. Export MASTER file (all fields, all records)
# ============================================================================

model_suffix = MODEL_CHOICE.replace("-", "_").replace(".", "_") if has_llm_results else "deterministic"
master_filename = f"Master_Job_Analysis_{model_suffix}.csv"
master_path = os.path.join(OUTDIR, master_filename)

df_export.to_csv(master_path, index=False)
print(f"✅ Exported Master File: {master_filename}")
print(f"   Total records: {len(df_export)}")
print(f"   Total columns: {len(df_export.columns)}")

# ============================================================================
# 4. Create FOCUSED REVIEW report (key columns, high-priority records)
# ============================================================================

# Define focused columns for HR review
focused_cols = [
    # Identification
    "job_title_original",
    "job_title_key",
    "major_role_group",

    # Evidence
    "human_error_probability",
    "similarity_numeric",
    "alt_count",
    "Consensus",
    "top_match_role",
    "alt_roles_list",

    # Scoring
    "likelihood_error_score_0_5",
    "likelihood_band",
    "review_priority",

    # LLM Assessment
    "score_assessment",
    "confidence",
    "notes",

    # Additional context
    "confusion_risk_score",
    "confusion_risk_category"
]

# Keep only columns that exist
available_focused = [c for c in focused_cols if c in df_export.columns]

# Filter for review (Priority 1-7 = Critical through High band + LLM flags)
df_review = df_export[df_export['review_priority'] >= 4].copy()
df_review_focused = df_review[available_focused].sort_values('review_priority', ascending=False)

if len(df_review) > 0:
    review_filename = f"FOR_HUMAN_REVIEW_{model_suffix}.csv"
    review_path = os.path.join(OUTDIR, review_filename)
    df_review_focused.to_csv(review_path, index=False)

    print(f"\n✅ Exported Review Report: {review_filename}")
    print(f"   Records for review: {len(df_review)} / {len(df_export)} ({len(df_review)/len(df_export)*100:.1f}%)")

    # Priority breakdown
    print(f"\n   Priority Breakdown:")
    print(f"      Critical (9-10):    {(df_review['review_priority'] >= 9).sum()} records")
    print(f"      Very High (7-8):    {((df_review['review_priority'] >= 7) & (df_review['review_priority'] < 9)).sum()} records")
    print(f"      High (5-6):         {((df_review['review_priority'] >= 5) & (df_review['review_priority'] < 7)).sum()} records")
    print(f"      Moderate (4):       {(df_review['review_priority'] == 4).sum()} records")

    files.download(review_path)
else:
    print("\n📋 No records flagged for human review")

# ============================================================================
# 5. Summary Report
# ============================================================================

print("\n" + "=" * 60)
print("📊 SUMMARY REPORT")
print("=" * 60)
print(f"Total records processed:  {len(df_export)}")
print(f"LLM reviewed:             {df_export['llm_reviewed'].sum()}")

if has_llm_results:
    print(f"Judge model used:         {config['model_id']}")

print(f"\n📊 Risk Band Distribution:")
for band in ["Critical", "Very High", "High", "Moderate", "Low", "Very Low"]:
    count = (df_export["likelihood_band"] == band).sum()
    if count > 0:
        pct = count / len(df_export) * 100
        print(f"   {band:12}: {count:2} ({pct:5.1f}%)")

if has_llm_results:
    print(f"\n📊 LLM Assessment Results:")
    reviewed = df_export[df_export["llm_reviewed"]]
    if len(reviewed) > 0:
        print(f"  Appropriate: {(reviewed['score_assessment'] == 'appropriate').sum()}")
        print(f"  Too high:    {(reviewed['score_assessment'] == 'too_high').sum()}")
        print(f"  Too low:     {(reviewed['score_assessment'] == 'too_low').sum()}")
        print(f"  Unknown:     {(reviewed['score_assessment'] == 'unknown').sum()}")

# Check for new fields
if 'similarity_numeric' in df_export.columns:
    print(f"\n📊 Similarity Score Statistics:")
    print(f"   Min: {df_export['similarity_numeric'].min():.1f}%")
    print(f"   Max: {df_export['similarity_numeric'].max():.1f}%")
    print(f"   Avg: {df_export['similarity_numeric'].mean():.1f}%")

print("=" * 60)
print("\n⬇️ Downloading master file...")
files.download(master_path)
print("✅ Complete!")

🚀 Exporting Hybrid Results (Deterministic + claude-sonnet-4.5)

✅ Exported Master File: Master_Job_Analysis_claude_sonnet_4_5.csv
   Total records: 43
   Total columns: 33

✅ Exported Review Report: FOR_HUMAN_REVIEW_claude_sonnet_4_5.csv
   Records for review: 21 / 43 (48.8%)

   Priority Breakdown:
      Critical (9-10):    1 records
      Very High (7-8):    6 records
      High (5-6):         13 records
      Moderate (4):       1 records


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 SUMMARY REPORT
Total records processed:  43
LLM reviewed:             31
Judge model used:         claude-sonnet-4-5-20250929

📊 Risk Band Distribution:
   Critical    :  9 ( 20.9%)
   Very High   :  6 ( 14.0%)
   High        :  6 ( 14.0%)
   Moderate    : 10 ( 23.3%)
   Low         : 10 ( 23.3%)
   Very Low    :  2 (  4.7%)

📊 LLM Assessment Results:
  Appropriate: 27
  Too high:    4
  Too low:     0
  Unknown:     0

📊 Similarity Score Statistics:
   Min: 29.4%
   Max: 88.2%
   Avg: 52.9%

⬇️ Downloading master file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Complete!


---
## 🔍 Validation & Analysis (Optional)

Run these cells to analyze the quality of results and identify potential issues.


In [17]:
# ==== Optional: Quality Analysis ====

if has_llm_results:
    print("🔍 QUALITY ANALYSIS\n")

    # Analysis 1: Score vs Assessment Alignment
    print("1️⃣ Score vs Assessment Alignment Check:")
    print("-" * 40)

    reviewed = df_export[df_export["llm_reviewed"]].copy()

    # High scores marked as "too_high"
    high_score_flagged = reviewed[
        (reviewed["likelihood_error_score_0_5"] >= 2.0) &
        (reviewed["score_assessment"] == "too_high") &
        (reviewed["alt_count"] >= 2)
    ]

    if len(high_score_flagged) > 0:
        print(f"⚠️ {len(high_score_flagged)} high scores flagged as 'too_high' despite multiple alternatives:")
        display(high_score_flagged[["job_title_key", "likelihood_error_score_0_5",
                                    "alt_count", "confusion_risk_score", "notes"]].head())
    else:
        print("✅ No suspicious 'too_high' flags")

    # Low scores marked as "too_low"
    print("\n2️⃣ Low Score Validation:")
    print("-" * 40)
    low_score_flagged = reviewed[
        (reviewed["likelihood_error_score_0_5"] < 2.0) &
        (reviewed["score_assessment"] == "too_low")
    ]

    if len(low_score_flagged) > 0:
        print(f"⚠️ {len(low_score_flagged)} low scores flagged as 'too_low':")
        display(low_score_flagged[["job_title_key", "likelihood_error_score_0_5",
                                   "alt_count", "confusion_risk_score", "notes"]].head())
    else:
        print("✅ No low scores flagged as 'too_low'")

    # Confidence distribution by assessment
    print("\n3️⃣ Confidence by Assessment Type:")
    print("-" * 40)
    conf_pivot = reviewed.groupby(["score_assessment", "confidence"]).size().unstack(fill_value=0)
    display(conf_pivot)

    # Hedging language detection
    print("\n4️⃣ Justification Quality Flags:")
    print("-" * 40)
    print(f"Hedging language detected: {reviewed['hedging_language'].sum()}")
    print(f"Title-only reasoning:      {reviewed['title_only_reasoning'].sum()}")
    print(f"Competing roles mentioned: {reviewed['mentions_competing_role_terms'].sum()}")

else:
    print("ℹ️ Run LLM evaluation first to see quality analysis")


🔍 QUALITY ANALYSIS

1️⃣ Score vs Assessment Alignment Check:
----------------------------------------
⚠️ 4 high scores flagged as 'too_high' despite multiple alternatives:


,job_title_key,likelihood_error_score_0_5,alt_count,confusion_risk_score,notes
7,assistant physical therapist,4.64,4,0,Score 4.64 excessive for healthcare assistant ...
19,chief financial officer,4.58,7,0,Score 4.58 inappropriate for CFO. Despite 7 al...
24,classroom management specialist,5.00,10,2,"Score 5.0 justified by 10 alternatives, but ju..."
28,assistant extended learning program supervisor,3.66,5,2,"Justification argues FOR Coordinator, but clas..."



2️⃣ Low Score Validation:
----------------------------------------
✅ No low scores flagged as 'too_low'

3️⃣ Confidence by Assessment Type:
----------------------------------------


confidence,high,medium
score_assessment,,
appropriate,27,0
too_high,3,1



4️⃣ Justification Quality Flags:
----------------------------------------
Hedging language detected: 1
Title-only reasoning:      0
Competing roles mentioned: 26
